[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nchernia/festival-multiomics-workshop/blob/main/rna_annotation.ipynb)

# RNA Cell-Type Annotation — self-paced notebook

This notebook walks through cell-type annotation on the RNA half of the multiome. The main workshop notebook starts from the *result* of this annotation; here you can dig into **how** it's built — module scoring, hierarchical lineage assignment, CD4/CD8 sub-clustering, and validation via differential expression.

Runtime: ~15 minutes if you step through.

---

In [ ]:
%%capture
!pip install -q scanpy muon 'anndata>=0.10' leidenalg igraph

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import muon as mu
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))
sc.settings.verbosity = 1

In [ ]:
# Load workshop data. The .h5mu carries a pre-computed cell_type from the
# preprocessing pipeline — we stash it as 'cell_type_ref' and re-derive a fresh
# annotation so you can compare your result against the baked-in labels.
import os, urllib.request

DATA = "pbmc_10k_multiome_workshop.h5mu"
if not os.path.exists(DATA):
    url = os.environ.get("DATA_URL", "workshop_data/" + DATA)
    if url.startswith("http"):
        print(f"Downloading from {url} ...")
        urllib.request.urlretrieve(url, DATA)
        print("Done!")
    else:
        DATA = url

mdata = mu.read(DATA)
rna = mdata.mod['rna']
if 'cell_type' in rna.obs.columns:
    rna.obs['cell_type_ref'] = rna.obs['cell_type'].astype(str)
    del rna.obs['cell_type']
print(f"rna: {rna.n_obs:,} cells x {rna.n_vars:,} genes")

## Cell-type annotation by marker module scoring

The RNA data has already been QC'd, normalized, and clustered with Leiden.
Our job: **assign cell type labels** to each cluster using marker gene module scores.

**Module scoring** (`sc.tl.score_genes`) computes a per-cell enrichment score for a set of marker genes, controlled against a random background. For each cluster, we assign the cell type whose markers score highest on average.

> **A note on the clustering:** Leiden at resolution 1.0 deliberately *over-partitions* — you'll see, e.g., B cells split across two or three clusters. That's expected: resolution 1.0 picks up subtle substructure (and some noise). We annotate at the *cluster* level and let multiple clusters share a label, rather than trying to tune the resolution so every cell type is exactly one cluster.

In [ ]:
# First, let's see what the unsupervised clustering looks like
sc.pl.umap(rna, color="leiden", legend_loc="on data", title="Leiden Clusters")

### Where do PBMC marker genes come from?

The marker dictionary below didn't come from this dataset — it's the same panel of **canonical immunology markers** that immunologists have used for decades:

- `CD3D/CD3E/CD3G` — the T-cell receptor complex; strong expression of these genes is characteristic of T cells
- `CD4`, `CD8A/CD8B` — CD4/CD8 co-receptors that name the two main T-cell lineages
- `MS4A1` (= CD20), `CD79A` — B-cell receptor machinery
- `CD14`, `FCGR3A` (= CD16) — flow markers that name monocyte subsets
- `NKG7`, `GNLY` — cytotoxic lymphocyte markers (NK cells and effector CD8s)

Many of the same lineage markers used in flow cytometry are also informative in scRNA-seq, although scRNA-seq measures mRNA rather than protein abundance.

**Modern source for curated panels:** the [Azimuth PBMC reference](https://azimuth.hubmapconsortium.org/references/human_pbmc/) (Satija lab) — their hierarchical annotation is the de-facto standard. This workshop's marker list is a simplified version of that reference.

**Two-pass annotation: lineage first, then T-cell subtype**

Closely related populations like CD4 and CD8 T cells share many markers (`CD3D`, `CD3E`, `TRAC`). And the markers we *want* for CD4 (e.g. `IL7R`, `LEF1`, `TCF7`) are mostly pan-memory or pan-naive T markers — they're expressed almost as strongly in CD8 T cells. `sc.tl.score_genes` then finds CD4 score > CD8 score in every T cluster, hiding real CD8 cells.

**Solution: hierarchical annotation.**

1. **Broad lineage** (`BROAD_MARKERS`): unambiguous lineage markers. Pick the broad type per cluster.
2. **T-cell subcluster + CD8 threshold**: subset to the T-cell clusters, re-run leiden, then label each sub-cluster by **mean `CD8A`+`CD8B` expression**. We don't score CD4 directly — in droplet scRNA-seq, CD4 transcript detection is often sparse, so PBMC workflows commonly identify CD4 T cells operationally as non-cytotoxic T-cell clusters lacking strong CD8 programs.

This hierarchical strategy is widely used for closely related cell-type families.

In [ ]:
# Two marker dicts:
#   BROAD_MARKERS picks the lineage (T/NK/B/Mono/DC) per cluster.
#   PBMC_MARKERS is a richer panel kept for the dotplot below.

BROAD_MARKERS = {
    "T cell":         ["CD3D", "CD3E", "CD3G", "TRAC", "TRBC2"],
    "NK cell":        ["NKG7", "GNLY", "KLRD1", "PRF1", "TYROBP"],
    "B cell":         ["MS4A1", "CD79A", "BANK1", "CD74", "HLA-DRA"],
    "CD14 Monocyte":  ["CD14", "VCAN", "FCN1", "S100A9", "LYZ"],
    "CD16 Monocyte":  ["FCGR3A", "MS4A7", "LST1", "IFITM3", "SAT1"],
    "Dendritic cell": ["FCER1A", "CD1C", "CLEC10A", "CST3", "HLA-DRA"],
}

PBMC_MARKERS = {
    "CD4 T cell":     ["CD3D", "CD3E", "IL7R", "CD4", "LEF1", "MAL"],
    "CD8 T cell":     ["CD3D", "CD3E", "CD8A", "CD8B", "GZMK", "NKG7"],
    "NK cell":        ["NKG7", "GNLY", "KLRD1", "KLRB1", "NCAM1", "PRF1"],
    "B cell":         ["MS4A1", "CD79A", "CD79B", "CD19", "PAX5", "BANK1"],
    "CD14 Monocyte":  ["CD14", "LYZ", "S100A8", "S100A9", "VCAN", "FCN1"],
    "CD16 Monocyte":  ["FCGR3A", "MS4A7", "LST1", "LILRB2", "IFITM3"],
    "Dendritic cell": ["FCER1A", "CST3", "CLEC10A", "CD1C", "ENHO"],
}

print("Broad lineage markers:")
for ct, genes in BROAD_MARKERS.items():
    found = [g for g in genes if g in rna.var_names]
    print(f"  {ct}: {len(found)}/{len(genes)} found")

In [ ]:
# Score each cell against the broad lineage panels.
for ct, genes in BROAD_MARKERS.items():
    available = [g for g in genes if g in rna.var_names]
    sc.tl.score_genes(rna, gene_list=available, score_name=f"score_{ct}")

print("Module scores computed:", [f"score_{ct}" for ct in BROAD_MARKERS])

In [ ]:
# Where do the broad-lineage module scores land on the UMAP?
score_keys = [f"score_{ct}" for ct in BROAD_MARKERS]
sc.pl.umap(rna, color=score_keys, ncols=3, cmap="viridis", frameon=False, wspace=0.25)

In [ ]:
import scipy.sparse as sp

# --- Pass 1: each leiden cluster -> broad lineage ---
clusters = sorted(rna.obs["leiden"].unique(), key=int)
broad_mat = pd.DataFrame(index=clusters, columns=list(BROAD_MARKERS), dtype=float)
for cl in clusters:
    mask = rna.obs["leiden"] == cl
    for ct in BROAD_MARKERS:
        broad_mat.loc[cl, ct] = rna.obs.loc[mask, f"score_{ct}"].mean()
cluster_to_broad = broad_mat.idxmax(axis=1).to_dict()
rna.obs["broad_type"] = rna.obs["leiden"].map(cluster_to_broad).astype("category")

# --- Pass 2: subcluster T cells, then CD4 vs CD8 by CD8A+CD8B expression ---
final = dict(cluster_to_broad)
t_mask = rna.obs["broad_type"] == "T cell"
if t_mask.sum() > 0:
    t_rna = rna[t_mask].copy()
    sc.pp.neighbors(t_rna, n_pcs=30)
    sc.tl.leiden(t_rna, resolution=1.0, key_added="t_sub")
    print(f"T-cell sub-clusters: {t_rna.obs['t_sub'].nunique()}")

    cd8_genes = [g for g in ("CD8A", "CD8B") if g in t_rna.var_names]
    X = t_rna[:, cd8_genes].X
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    t_rna.obs["cd8_expr"] = X.mean(axis=1)

    sub_to_label = {}
    for sub in sorted(t_rna.obs["t_sub"].unique(), key=int):
        m = (t_rna.obs["t_sub"] == sub).values
        mean_cd8 = t_rna.obs.loc[m, "cd8_expr"].mean()
        sub_to_label[sub] = "CD8 T cell" if mean_cd8 > 0.3 else "CD4 T cell"
        print(f"  T sub {sub:>2} ({int(m.sum()):>4} cells): mean CD8A+CD8B = {mean_cd8:.2f} -> {sub_to_label[sub]}")
    t_rna.obs["t_label"] = t_rna.obs["t_sub"].map(sub_to_label)

    for cl in [c for c in clusters if cluster_to_broad[c] == "T cell"]:
        in_cl = (t_rna.obs["leiden"] == cl).values
        labels = t_rna.obs.loc[in_cl, "t_label"]
        final[cl] = labels.mode().iloc[0] if len(labels) else "CD4 T cell"

# --- Materialise final labels ---
rna.obs["cell_type"] = rna.obs["leiden"].map(final).astype("category")
rna.obs["cluster_label"] = rna.obs.apply(
    lambda r: f"{r['leiden']}: {final[r['leiden']]}", axis=1
).astype("category")

table = pd.DataFrame({
    "cluster": clusters,
    "broad":   [cluster_to_broad[c] for c in clusters],
    "final":   [final[c]            for c in clusters],
    "n_cells": [int((rna.obs["leiden"] == c).sum()) for c in clusters],
})
print()
print(table.to_string(index=False))
print()
print("Per-type totals:")
print(rna.obs["cell_type"].value_counts().to_string())

In [ ]:
# Visualize: leiden clusters, combined cluster:celltype, and pure cell type.
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sc.pl.umap(rna, color="leiden", ax=axes[0], show=False,
           title="Leiden Clusters", legend_loc="on data", legend_fontsize=8)
sc.pl.umap(rna, color="cluster_label", ax=axes[1], show=False,
           title="Cluster ID + Annotation", legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(rna, color="cell_type", ax=axes[2], show=False,
           title="Cell Type", legend_loc="right margin", legend_fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Dotplot: marker gene expression by cell type annotation.
markers_present = {ct: [g for g in genes if g in rna.var_names]
                   for ct, genes in PBMC_MARKERS.items()}
markers_present = {ct: gs for ct, gs in markers_present.items() if gs}

sc.pl.dotplot(rna, var_names=markers_present, groupby="cell_type",
              standard_scale="var", title="Marker Genes by Annotated Cell Type")

### Cross-check: data-driven differentially expressed genes

The marker scores were *supervised* — we told scanpy which genes to look at. Now ask the data which genes are most upregulated in each annotated cell type, with no prior knowledge. The two views should agree if the annotation is good.

In [ ]:
# Top DEGs per cell type via Wilcoxon rank-sum test
sc.tl.rank_genes_groups(rna, groupby="cell_type", method="wilcoxon",
                        n_genes=200, use_raw=False)

top_per_type = pd.DataFrame({
    grp: sc.get.rank_genes_groups_df(rna, group=grp).head(5)["names"].values
    for grp in rna.obs["cell_type"].cat.categories
})
print("Top 5 DEGs per annotated cell type:")
print(top_per_type.to_string(index=False))

sc.pl.rank_genes_groups_dotplot(rna, n_genes=3, standard_scale="var", groupby="cell_type")

### Compare to the pre-baked annotation

The `.h5mu` was preprocessed with the same hierarchical annotation, stashed as `cell_type_ref`. Let's see how closely the labels you just derived agree with the reference.

In [ ]:
ct = pd.DataFrame({
    'derived':   rna.obs['cell_type'].astype(str),
    'reference': rna.obs['cell_type_ref'].astype(str),
})
agree = (ct['derived'] == ct['reference']).mean()
print(f"Label agreement: {agree:.1%}  ({(ct['derived']==ct['reference']).sum():,}/{len(ct):,} cells)")
print()
print(pd.crosstab(ct['derived'], ct['reference']).to_string())